# Validação dos relacionamentos entre entidades

## Introdução

Após a validação das entidades, de seus atributos e dos identificadores conceituais, esta etapa verifica se os relacionamentos propostos para o modelo conceitual possuem sustentação estrutural nos dados do projeto.

A análise concentra-se na integridade referencial observada entre os arquivos do *Brazilian E-Commerce Public Dataset by Olist*. O objetivo não é antecipar a definição formal das cardinalidades, mas confirmar que as conexões entre as entidades possuem correspondência efetiva no conjunto de dados e identificar eventuais registros órfãos ou particularidades estruturais que precisem ser consideradas na modelagem.

A entidade **Geolocalização** recebe tratamento específico, pois o prefixo de CEP é utilizado como atributo de associação, mas não constitui identificador único de uma ocorrência geográfica.

## Objetivos

- validar os oito relacionamentos conceituais previamente identificados;
- verificar a existência de correspondência entre identificadores relacionados;
- quantificar registros ou valores distintos sem correspondência na entidade de referência;
- calcular a taxa de cobertura referencial de cada relacionamento;
- produzir exemplos de valores órfãos quando existirem;
- analisar a multiplicidade de registros por prefixo de CEP na entidade Geolocalização;
- registrar evidências que sustentem as decisões posteriores de modelagem conceitual.

## 1. Critérios de validação

Cada relacionamento será avaliado segundo os seguintes critérios:

| Critério | Pergunta de validação |
|---|---|
| Correspondência | Os valores utilizados na associação existem na entidade de referência? |
| Orfandade | Existem valores no lado dependente sem correspondência no lado de referência? |
| Cobertura | Qual proporção dos valores distintos dependentes encontra correspondência? |
| Coerência | A associação observada é compatível com a semântica definida para o domínio? |
| Granularidade | O relacionamento preserva adequadamente o nível de detalhe das entidades? |
| Particularidades | Existem situações que exigem tratamento específico na etapa conceitual ou lógica? |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Localização e carregamento dos dados

O notebook procura o diretório `data/raw` a partir da raiz do projeto e, alternativamente, `../../data/raw` quando executado a partir de `notebooks/data_understanding`.

Os arquivos brutos não são modificados nesta etapa.

In [2]:
data_directories = (
    Path("data/raw"),
    Path("../../data/raw"),
)

data_dir = next((path for path in data_directories if path.is_dir()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Diretório data/raw não encontrado. Consulte data/README.md para obter os dados."
    )

data_dir.resolve()

PosixPath('/home/lucas/workspace/pessoal/ecommerce-analytics-data-model/data/raw')

In [3]:
arquivos_entidades = {
    "Clientes": "olist_customers_dataset.csv",
    "Pedidos": "olist_orders_dataset.csv",
    "Itens do Pedido": "olist_order_items_dataset.csv",
    "Produtos": "olist_products_dataset.csv",
    "Vendedores": "olist_sellers_dataset.csv",
    "Pagamentos": "olist_order_payments_dataset.csv",
    "Avaliações": "olist_order_reviews_dataset.csv",
    "Geolocalização": "olist_geolocation_dataset.csv",
}

entidades = {
    nome: pd.read_csv(data_dir / arquivo)
    for nome, arquivo in arquivos_entidades.items()
}

pd.DataFrame(
    {
        "Entidade": entidades.keys(),
        "Registros": [len(df) for df in entidades.values()],
        "Colunas": [len(df.columns) for df in entidades.values()],
    }
)

,Entidade,Registros,Colunas
0,Clientes,99441,5
1,Pedidos,99441,8
2,Itens do Pedido,112650,7
3,Produtos,32951,9
4,Vendedores,3095,4
5,Pagamentos,103886,5
6,Avaliações,99224,7
7,Geolocalização,1000163,5


## 3. Relacionamentos submetidos à validação

Os seguintes relacionamentos foram previamente aprovados em nível conceitual:

1. Cliente **realiza** Pedido;
2. Pedido **possui** Item do Pedido;
3. Item do Pedido **refere-se a** Produto;
4. Vendedor **vende** Item do Pedido;
5. Pedido **possui** Pagamento;
6. Pedido **recebe** Avaliação;
7. Cliente **associa-se geograficamente a** Geolocalização;
8. Vendedor **associa-se geograficamente a** Geolocalização.

Nesta etapa, a análise concentra-se exclusivamente na existência e na consistência das associações. A cardinalidade formal será tratada em etapa posterior.

In [4]:
relacionamentos = [
    {
        "relacionamento": "Cliente realiza Pedido",
        "referencia_entidade": "Clientes",
        "referencia_coluna": "customer_id",
        "dependente_entidade": "Pedidos",
        "dependente_coluna": "customer_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Pedido possui Item do Pedido",
        "referencia_entidade": "Pedidos",
        "referencia_coluna": "order_id",
        "dependente_entidade": "Itens do Pedido",
        "dependente_coluna": "order_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Item do Pedido refere-se a Produto",
        "referencia_entidade": "Produtos",
        "referencia_coluna": "product_id",
        "dependente_entidade": "Itens do Pedido",
        "dependente_coluna": "product_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Vendedor vende Item do Pedido",
        "referencia_entidade": "Vendedores",
        "referencia_coluna": "seller_id",
        "dependente_entidade": "Itens do Pedido",
        "dependente_coluna": "seller_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Pedido possui Pagamento",
        "referencia_entidade": "Pedidos",
        "referencia_coluna": "order_id",
        "dependente_entidade": "Pagamentos",
        "dependente_coluna": "order_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Pedido recebe Avaliação",
        "referencia_entidade": "Pedidos",
        "referencia_coluna": "order_id",
        "dependente_entidade": "Avaliações",
        "dependente_coluna": "order_id",
        "natureza": "Identificador direto",
    },
    {
        "relacionamento": "Cliente associa-se geograficamente a Geolocalização",
        "referencia_entidade": "Geolocalização",
        "referencia_coluna": "geolocation_zip_code_prefix",
        "dependente_entidade": "Clientes",
        "dependente_coluna": "customer_zip_code_prefix",
        "natureza": "Associação geográfica por prefixo de CEP",
    },
    {
        "relacionamento": "Vendedor associa-se geograficamente a Geolocalização",
        "referencia_entidade": "Geolocalização",
        "referencia_coluna": "geolocation_zip_code_prefix",
        "dependente_entidade": "Vendedores",
        "dependente_coluna": "seller_zip_code_prefix",
        "natureza": "Associação geográfica por prefixo de CEP",
    },
]

pd.DataFrame(relacionamentos)

,relacionamento,referencia_entidade,referencia_coluna,dependente_entidade,dependente_coluna,natureza
0,Cliente realiza Pedido,Clientes,customer_id,Pedidos,customer_id,Identificador direto
1,Pedido possui Item do Pedido,Pedidos,order_id,Itens do Pedido,order_id,Identificador direto
2,Item do Pedido refere-se a Produto,Produtos,product_id,Itens do Pedido,product_id,Identificador direto
3,Vendedor vende Item do Pedido,Vendedores,seller_id,Itens do Pedido,seller_id,Identificador direto
4,Pedido possui Pagamento,Pedidos,order_id,Pagamentos,order_id,Identificador direto
5,Pedido recebe Avaliação,Pedidos,order_id,Avaliações,order_id,Identificador direto
6,Cliente associa-se geograficamente a Geolocalização,Geolocalização,geolocation_zip_code_prefix,Clientes,customer_zip_code_prefix,Associação geográfica por prefixo de CEP
7,Vendedor associa-se geograficamente a Geolocalização,Geolocalização,geolocation_zip_code_prefix,Vendedores,seller_zip_code_prefix,Associação geográfica por prefixo de CEP


## 4. Funções de validação

A validação considera **valores distintos** no lado dependente. Essa abordagem evita que uma chave utilizada muitas vezes distorça a taxa de cobertura.

Para cada relacionamento são calculados:

- quantidade de valores distintos na entidade dependente;
- quantidade de valores distintos na entidade de referência;
- quantidade de valores dependentes com correspondência;
- quantidade de valores dependentes órfãos;
- taxa de cobertura referencial;
- quantidade de valores ausentes no lado dependente;
- exemplos de valores órfãos, quando existentes.

In [5]:
def normalizar_serie_chave(serie: pd.Series) -> pd.Series:
    """Remove valores ausentes e preserva a representação dos identificadores."""
    return serie.dropna()


def validar_relacionamento(
    referencia: pd.DataFrame,
    coluna_referencia: str,
    dependente: pd.DataFrame,
    coluna_dependente: str,
    nome_relacionamento: str,
    natureza: str,
    limite_exemplos: int = 10,
):
    valores_referencia = pd.Index(
        normalizar_serie_chave(referencia[coluna_referencia]).unique()
    )
    valores_dependentes = pd.Index(
        normalizar_serie_chave(dependente[coluna_dependente]).unique()
    )

    correspondentes = valores_dependentes.intersection(valores_referencia)
    orfaos = valores_dependentes.difference(valores_referencia)

    total_dependentes = len(valores_dependentes)
    taxa_cobertura = (
        len(correspondentes) / total_dependentes
        if total_dependentes
        else np.nan
    )

    resultado = {
        "Relacionamento": nome_relacionamento,
        "Natureza": natureza,
        "Valores distintos — referência": len(valores_referencia),
        "Valores distintos — dependente": total_dependentes,
        "Correspondentes": len(correspondentes),
        "Órfãos": len(orfaos),
        "Cobertura (%)": taxa_cobertura * 100 if pd.notna(taxa_cobertura) else np.nan,
        "Ausentes — dependente": int(dependente[coluna_dependente].isna().sum()),
    }

    detalhes = {
        "orfaos": list(orfaos[:limite_exemplos]),
        "total_orfaos": len(orfaos),
    }

    return resultado, detalhes

## 5. Validação referencial consolidada

In [6]:
resultados = []
detalhes_orfaos = {}

for rel in relacionamentos:
    resultado, detalhes = validar_relacionamento(
        referencia=entidades[rel["referencia_entidade"]],
        coluna_referencia=rel["referencia_coluna"],
        dependente=entidades[rel["dependente_entidade"]],
        coluna_dependente=rel["dependente_coluna"],
        nome_relacionamento=rel["relacionamento"],
        natureza=rel["natureza"],
    )
    resultados.append(resultado)
    detalhes_orfaos[rel["relacionamento"]] = detalhes

df_validacao = pd.DataFrame(resultados)

df_validacao.style.format(
    {
        "Cobertura (%)": "{:.2f}%",
    }
)

,Relacionamento,Natureza,Valores distintos — referência,Valores distintos — dependente,Correspondentes,Órfãos,Cobertura (%),Ausentes — dependente
0,Cliente realiza Pedido,Identificador direto,99441,99441,99441,0,100.00%,0
1,Pedido possui Item do Pedido,Identificador direto,99441,98666,98666,0,100.00%,0
2,Item do Pedido refere-se a Produto,Identificador direto,32951,32951,32951,0,100.00%,0
3,Vendedor vende Item do Pedido,Identificador direto,3095,3095,3095,0,100.00%,0
4,Pedido possui Pagamento,Identificador direto,99441,99440,99440,0,100.00%,0
5,Pedido recebe Avaliação,Identificador direto,99441,98673,98673,0,100.00%,0
6,Cliente associa-se geograficamente a Geolocalização,Associação geográfica por prefixo de CEP,19015,14994,14837,157,98.95%,0
7,Vendedor associa-se geograficamente a Geolocalização,Associação geográfica por prefixo de CEP,19015,2246,2239,7,99.69%,0


### 5.1 Valores órfãos

A presença de valores órfãos não invalida automaticamente um relacionamento conceitual. Quando existirem, eles devem ser interpretados como evidência de possível problema de qualidade, cobertura ou integridade do conjunto de dados e avaliados antes da modelagem lógica e física.

In [7]:
linhas_orfaos = []

for relacionamento, detalhes in detalhes_orfaos.items():
    linhas_orfaos.append(
        {
            "Relacionamento": relacionamento,
            "Total de órfãos": detalhes["total_orfaos"],
            "Exemplos": detalhes["orfaos"] if detalhes["orfaos"] else "Nenhum",
        }
    )

df_orfaos = pd.DataFrame(linhas_orfaos)
df_orfaos

,Relacionamento,Total de órfãos,Exemplos
0,Cliente realiza Pedido,0,Nenhum
1,Pedido possui Item do Pedido,0,Nenhum
2,Item do Pedido refere-se a Produto,0,Nenhum
3,Vendedor vende Item do Pedido,0,Nenhum
4,Pedido possui Pagamento,0,Nenhum
5,Pedido recebe Avaliação,0,Nenhum
6,Cliente associa-se geograficamente a Geolocalização,157,"[2140, 6930, 7412, 7430, 7729, 7784, 8342, 8980, 11547, 12332]"
7,Vendedor associa-se geograficamente a Geolocalização,7,"[2285, 7412, 37708, 71551, 72580, 82040, 91901]"


## 6. Análise específica da entidade Geolocalização

O prefixo de CEP não deve ser interpretado como chave primária da entidade Geolocalização.

A análise abaixo mede quantas observações geográficas existem para cada `geolocation_zip_code_prefix`. Essa multiplicidade é relevante porque Clientes e Vendedores são associados à base geográfica por prefixo de CEP, mas um mesmo prefixo pode corresponder a várias coordenadas registradas.

In [8]:
geolocalizacao = entidades["Geolocalização"]

multiplicidade_cep = (
    geolocalizacao
    .groupby("geolocation_zip_code_prefix", dropna=False)
    .size()
    .rename("quantidade_registros")
)

resumo_multiplicidade = pd.DataFrame(
    {
        "Métrica": [
            "Prefixos distintos",
            "Mínimo de registros por prefixo",
            "Mediana de registros por prefixo",
            "Média de registros por prefixo",
            "Máximo de registros por prefixo",
            "Prefixos com múltiplas ocorrências",
            "Percentual de prefixos com múltiplas ocorrências",
        ],
        "Valor": [
            int(multiplicidade_cep.shape[0]),
            int(multiplicidade_cep.min()),
            float(multiplicidade_cep.median()),
            float(multiplicidade_cep.mean()),
            int(multiplicidade_cep.max()),
            int((multiplicidade_cep > 1).sum()),
            float((multiplicidade_cep > 1).mean() * 100),
        ],
    }
)

resumo_multiplicidade

,Métrica,Valor
0,Prefixos distintos,19015.000000
1,Mínimo de registros por prefixo,1.000000
2,Mediana de registros por prefixo,29.000000
3,Média de registros por prefixo,52.598633
4,Máximo de registros por prefixo,1146.000000
5,Prefixos com múltiplas ocorrências,17972.000000
6,Percentual de prefixos com múltiplas ocorrências,94.514857


In [9]:
(
    multiplicidade_cep
    .sort_values(ascending=False)
    .head(20)
    .rename_axis("geolocation_zip_code_prefix")
    .reset_index()
)

,geolocation_zip_code_prefix,quantidade_registros
0,24220,1146
1,24230,1102
2,38400,965
3,35500,907
4,11680,879
5,22631,832
6,30140,810
7,11740,788
8,38408,773
9,28970,743


### 6.1 Cobertura geográfica de Clientes e Vendedores

As tabelas abaixo permitem observar diretamente quais prefixos utilizados por Clientes ou Vendedores não possuem correspondência na entidade Geolocalização.

In [10]:
for nome_relacionamento in [
    "Cliente associa-se geograficamente a Geolocalização",
    "Vendedor associa-se geograficamente a Geolocalização",
]:
    print(nome_relacionamento)
    print("-" * len(nome_relacionamento))
    detalhes = detalhes_orfaos[nome_relacionamento]
    print(f"Total de prefixos órfãos: {detalhes['total_orfaos']}")
    print(f"Exemplos: {detalhes['orfaos'] if detalhes['orfaos'] else 'Nenhum'}")
    print()

Cliente associa-se geograficamente a Geolocalização
---------------------------------------------------
Total de prefixos órfãos: 157
Exemplos: [2140, 6930, 7412, 7430, 7729, 7784, 8342, 8980, 11547, 12332]

Vendedor associa-se geograficamente a Geolocalização
----------------------------------------------------
Total de prefixos órfãos: 7
Exemplos: [2285, 7412, 37708, 71551, 72580, 82040, 91901]



## 7. Diagnóstico automático para apoio à interpretação

A classificação abaixo serve apenas como apoio à leitura dos resultados. A decisão conceitual continua dependendo da análise semântica do domínio.

- **Cobertura integral:** não foram encontrados valores órfãos entre os identificadores distintos;
- **Cobertura parcial:** existem valores órfãos e eles devem ser investigados;
- **Sem valores dependentes:** situação excepcional em que não existem valores distintos para validar.

In [11]:
def classificar_cobertura(row):
    if row["Valores distintos — dependente"] == 0:
        return "Sem valores dependentes"
    if row["Órfãos"] == 0:
        return "Cobertura integral"
    return "Cobertura parcial — investigar órfãos"


df_diagnostico = df_validacao.copy()
df_diagnostico["Diagnóstico"] = df_diagnostico.apply(
    classificar_cobertura,
    axis=1,
)

df_diagnostico[
    [
        "Relacionamento",
        "Natureza",
        "Valores distintos — dependente",
        "Correspondentes",
        "Órfãos",
        "Cobertura (%)",
        "Diagnóstico",
    ]
].style.format({"Cobertura (%)": "{:.2f}%"})

,Relacionamento,Natureza,Valores distintos — dependente,Correspondentes,Órfãos,Cobertura (%),Diagnóstico
0,Cliente realiza Pedido,Identificador direto,99441,99441,0,100.00%,Cobertura integral
1,Pedido possui Item do Pedido,Identificador direto,98666,98666,0,100.00%,Cobertura integral
2,Item do Pedido refere-se a Produto,Identificador direto,32951,32951,0,100.00%,Cobertura integral
3,Vendedor vende Item do Pedido,Identificador direto,3095,3095,0,100.00%,Cobertura integral
4,Pedido possui Pagamento,Identificador direto,99440,99440,0,100.00%,Cobertura integral
5,Pedido recebe Avaliação,Identificador direto,98673,98673,0,100.00%,Cobertura integral
6,Cliente associa-se geograficamente a Geolocalização,Associação geográfica por prefixo de CEP,14994,14837,157,98.95%,Cobertura parcial — investigar órfãos
7,Vendedor associa-se geograficamente a Geolocalização,Associação geográfica por prefixo de CEP,2246,2239,7,99.69%,Cobertura parcial — investigar órfãos


## 8. Verificação dos relacionamentos não diretos

Os relacionamentos abaixo **não** são tratados como associações diretas do modelo conceitual, pois podem ser alcançados por meio de entidades intermediárias ou não possuem sustentação semântica direta no dataset:

- Cliente — Produto;
- Cliente — Pagamento;
- Cliente — Avaliação;
- Cliente — Vendedor;
- Pedido — Produto;
- Pedido — Vendedor;
- Produto — Vendedor;
- Produto — Pagamento;
- Produto — Avaliação;
- Vendedor — Pagamento;
- Vendedor — Avaliação;
- Pagamento — Avaliação.

A exclusão dessas associações evita redundância estrutural e preserva a granularidade das entidades intermediárias, especialmente **Item do Pedido** e **Pedido**.

## 9. Resultado da etapa

Após a execução das células anteriores, esta seção deve ser utilizada para registrar a interpretação dos resultados observados.

### Questões para validação

1. Os seis relacionamentos baseados em identificadores diretos apresentam cobertura integral?
2. Existem registros órfãos relevantes? Em caso afirmativo, tratam-se de problemas de qualidade dos dados ou de indícios de uma associação conceitual inadequada?
3. Qual é a cobertura dos prefixos de CEP de Clientes na entidade Geolocalização?
4. Qual é a cobertura dos prefixos de CEP de Vendedores na entidade Geolocalização?
5. A multiplicidade observada por prefixo de CEP confirma que `geolocation_zip_code_prefix` não pode assumir o papel de chave primária da entidade Geolocalização?
6. Algum resultado exige revisão dos oito relacionamentos conceituais previamente aprovados?

### Decisão

A decisão final sobre a aceitação dos relacionamentos deve ser registrada somente após a execução e interpretação dos resultados. Eventuais registros órfãos devem ser documentados como evidências de qualidade ou integridade dos dados, sem que isso implique automaticamente a rejeição do relacionamento conceitual.

## 10. Próxima etapa

Após a validação referencial e a aprovação dos relacionamentos, os resultados relevantes deverão ser incorporados ao documento de Modelagem Conceitual.

A etapa subsequente poderá então avançar para a definição formal das **cardinalidades e opcionalidades**, utilizando como base:

- as regras de negócio;
- a estrutura dos identificadores;
- a multiplicidade observada nos dados;
- as particularidades registradas durante esta validação.